# It Fits Random Labels Perfectly

**Topic:** Rademacher complexity &nbsp;·&nbsp; **Difficulty:** Medium

Rademacher complexity asks how well a hypothesis class can fit pure noise. If a
model can drive its training error to zero on random labels, then its capacity is
large enough that low training error carries no information about
generalization, which is the empirical heart of the 2017 "understanding deep
learning requires rethinking generalization" result.

Run the experiment at several model widths and watch the two curves separate.

## The data (created in the next cell)

- `X` `(400, 30)`: fixed features.
- `y_real`: labels that depend on `X`. `y_rand`: labels that do not.
- Model: ridge-regularized least-squares classification with a random ReLU
  feature expansion of width `m`.
- `widths = [10, 30, 100, 300, 1000, 3000]`.

## Your task

With **NumPy only**:

1. For each width, build random ReLU features `H = relu(X @ R)` with `R` having
   independent entries of standard deviation `1/sqrt(30)` (variance `1/30`), from
   `np.random.default_rng(300 + m)`.
2. Fit `w = (H^T H + 1e-6 I)^{-1} H^T y` for both label sets and record training
   accuracy (sign agreement) and, for the real labels only, test accuracy on a
   held-out set.
3. `emp_rademacher`: for each width, the mean over 20 random sign vectors
   `sigma` of `||H^T sigma|| / (n * ||sigma||)`, that is, the maximum of
   `(1/n) sigma^T H w` over unit-norm `w`, further divided by `||sigma||`.
   Draw the 20 vectors in turn as `rs.choice([-1.0, 1.0], n)` from one
   `rs = np.random.default_rng(777 + m)` per width.
4. Set `answers` to a dict with `"train_real"`, `"train_rand"`, `"test_real"`,
   `"emp_rademacher"` (length-6 each), and `"width_fits_noise"`, the smallest
   width reaching 99% training accuracy on random labels (`-1` if none).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks random-label training accuracy rises to near 1 (with the narrowest width
still below 0.8), that the empirical Rademacher value grows from the first width
to the last, and that test accuracy on real labels does not follow it.

In [ ]:
import numpy as np

rng = np.random.default_rng(5001)
n, d = 400, 30
X = rng.normal(0, 1, (n, d))
w_true = rng.normal(0, 1, d)
y_real = np.sign(X @ w_true + rng.normal(0, 0.5, n))
y_rand = rng.choice([-1.0, 1.0], n)

X_test = rng.normal(0, 1, (600, d))
y_test = np.sign(X_test @ w_true + rng.normal(0, 0.5, 600))
widths = [10, 30, 100, 300, 1000, 3000]
print(f"X {X.shape}; y_rand is independent of X by construction")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("train_real", [6], 5.3925, 2.33747449852909, 1e-06),
    ("train_rand", [6], 4.862500000000001, 2.553583550667904, 1e-06),
    ("test_real", [6], 4.471666666666667, 1.6217654997116209, 1e-06),
    ("emp_rademacher", [6], 0.215369034921289, 0.16966568903864201, 1e-06),
    ("width_fits_noise", [], 1000.0, -167.43801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "train_rand" in answers:
    _tr = np.asarray(answers["train_rand"], dtype=float)
    _te = np.asarray(answers["test_real"], dtype=float)
    _r = np.asarray(answers["emp_rademacher"], dtype=float)
    _memorize = _tr[-1] > 0.99 and _tr[0] < 0.8
    _rise = _r[-1] > _r[0]
    _test_ok = _te[-1] > 0.7
    print(f"  {'wide model fits noise':<24} {'ok' if _memorize else 'wrong'}  "
          f"({_tr[0]:.3f} at width {widths[0]} -> {_tr[-1]:.3f} at {widths[-1]})")
    print(f"  {'Rademacher grows':<24} {'ok' if _rise else 'wrong'}")
    print(f"  {'real test stays good':<24} {'ok' if _test_ok else 'wrong'}  "
          f"({_te[-1]:.3f})")
    _ok = _ok and _memorize and _rise and _test_ok

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Once the feature width exceeds the sample size the design matrix can interpolate
anything, so training accuracy on random labels hits 1 and stops being evidence of
anything at all. The empirical Rademacher term grows with width in step with that,
which is what makes it the right complexity notion here: unlike VC dimension it is
data-dependent, so it reflects the actual features rather than a worst case. Note
that test accuracy on real labels does *not* collapse. Capacity to memorize does
not by itself cause bad generalization, which is exactly the puzzle the paper
raised.

```python
import numpy as np

def fit(H, y):
    A = H.T @ H + 1e-6 * np.eye(H.shape[1])
    return np.linalg.solve(A, H.T @ y)

train_real, train_rand, test_real, rade = [], [], [], []
for m in widths:
    r = np.random.default_rng(300 + m)
    R = r.normal(0, 1 / np.sqrt(d), (d, m))
    H = np.maximum(X @ R, 0.0)
    Ht = np.maximum(X_test @ R, 0.0)

    w1 = fit(H, y_real)
    w2 = fit(H, y_rand)
    train_real.append(float((np.sign(H @ w1) == y_real).mean()))
    train_rand.append(float((np.sign(H @ w2) == y_rand).mean()))
    test_real.append(float((np.sign(Ht @ w1) == y_test).mean()))

    rs = np.random.default_rng(777 + m)
    vals = []
    for _ in range(20):
        sig = rs.choice([-1.0, 1.0], n)
        vals.append(float(np.linalg.norm(H.T @ sig) / (n * np.linalg.norm(sig))))
    rade.append(float(np.mean(vals)))

train_rand = np.array(train_rand)
hit = np.flatnonzero(train_rand >= 0.99)

answers = {
    "train_real": np.array(train_real),
    "train_rand": train_rand,
    "test_real": np.array(test_real),
    "emp_rademacher": np.array(rade),
    "width_fits_noise": float(widths[int(hit[0])]) if len(hit) else -1.0,
}
```